Build Production Pipeline

In [16]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

In [17]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW = ROOT / "data" / "raw"

MODELS = ROOT / "models"

In [18]:
train = pd.read_csv(
    RAW / "train.csv"
)

train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [19]:
X = train.drop(
    "SalePrice",
    axis=1
)

y = train["SalePrice"]

In [20]:
categorical_columns = X.select_dtypes(
    include="object"
).columns

numerical_columns = X.select_dtypes(
    exclude="object"
).columns

In [21]:
numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

In [22]:
categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

In [23]:
preprocessor = ColumnTransformer([

    (
        "num",

        numeric_transformer,

        numerical_columns
    ),

    (

        "cat",

        categorical_transformer,

        categorical_columns
    )

])

In [24]:
pipeline = Pipeline([

    (

        "preprocessor",

        preprocessor

    ),

    (

        "model",

        XGBRegressor(

            objective="reg:squarederror",

            random_state=42

        )

    )

])

In [25]:
pipeline.fit(
    X,
    y
)
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)

joblib.dump(
    pipeline,
    "models/house_price_pipeline.pkl"
)

print("Model saved successfully!")


Model saved successfully!


In [26]:
joblib.dump(

    pipeline,

    MODELS / "house_price_pipeline.pkl"
)

print("Pipeline saved successfully!")

Pipeline saved successfully!


In [27]:
pipeline.predict(

    X.iloc[[0]]
)

array([206510.95], dtype=float32)

In [28]:
import joblib

pipeline = joblib.load(
    "models/house_price_pipeline.pkl"
)

print(type(pipeline))

<class 'sklearn.pipeline.Pipeline'>


In [29]:
from pathlib import Path

print(Path.cwd())
print(list(Path(".").glob("**/*.pkl")))

C:\Users\USER\project\house-price-prediction\notebooks
[WindowsPath('models/house_price_pipeline.pkl')]


In [30]:
from pathlib import Path

print(Path("models/house_price_pipeline.pkl").exists())

True
